[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Gaurav14cs17/Multimodal-Deep-Learning/blob/main/01_Multimodal_Foundations/03_fusion_strategies/03_fusion_strategies.ipynb)

# 03. Fusion Strategies: How to Combine Modalities

**This notebook covers:**
- Early Fusion, Late Fusion, Cross-Modal Fusion — built from scratch
- When to use which strategy
- Cross-Attention: the most powerful fusion mechanism
- Gated fusion and attention-based pooling

---

In [ ]:
# ============================================================
#  Colab Setup (run this cell first if on Google Colab)
# ============================================================
import os

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    REPO_URL = "https://github.com/Gaurav14cs17/Multimodal-Deep-Learning.git"
    REPO_DIR = "/content/Multimodal-Deep-Learning"

    if not os.path.exists(REPO_DIR):
        !git clone {REPO_URL} {REPO_DIR}
        !pip install -q -r {REPO_DIR}/requirements.txt

    os.chdir(f"{REPO_DIR}/01_Multimodal_Foundations/03_fusion_strategies")
    os.makedirs(f"{REPO_DIR}/assets", exist_ok=True)
    print(f"Colab ready — working in {os.getcwd()}")
else:
    os.makedirs("../assets", exist_ok=True)

In [ ]:
import sys
sys.path.append('../..')

import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.patches import FancyBboxPatch
from utils.visualization import *
from utils.helpers import count_parameters

set_style()

In [ ]:
# Visualize all three fusion strategies side-by-side
fig = draw_fusion_comparison()
plt.savefig('../assets/fusion_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

![ViLBERT Cross-Attention Fusion — Lu et al. (2019)](../assets/paper_figure_fusion_patterns.png)

*Source: Lu et al. (2019) — "ViLBERT: Pretraining Task-Agnostic Visiolinguistic Representations" — [arXiv:1908.02265](https://arxiv.org/abs/1908.02265)*

*See also: VisualBERT (2019), Perceiver (2021), BLIP-2 (2023)*

## 1. Early Fusion

**Idea:** Concatenate raw inputs (or early features) BEFORE the main encoder.  
**When:** When modalities are similar in structure or you want maximum interaction.

### Early Fusion — Mathematical Formulation

In early fusion, tokens from all modalities are concatenated **before** joint processing:

$$h_{\text{early}} = \text{Transformer}([\mathbf{v}_1, \ldots, \mathbf{v}_N, \mathbf{t}_1, \ldots, \mathbf{t}_M])$$

where $\mathbf{v}_i \in \mathbb{R}^d$ are image patch embeddings and $\mathbf{t}_j \in \mathbb{R}^d$ are text token embeddings.

**Computational cost:** Self-attention over the combined sequence has complexity:

$$O\left((N + M)^2 \cdot d\right)$$

This is **quadratic in total sequence length** — if you have 196 image patches + 77 text tokens = 273 tokens, attention computes a $273 \times 273$ matrix at every layer. For long sequences this becomes prohibitively expensive, which is why most modern VLMs prefer cross-attention over early fusion.

In [ ]:
class EarlyFusion(nn.Module):
    """Concatenate image patches and text tokens, then process jointly."""
    def __init__(self, img_dim=128, txt_dim=128, hidden_dim=256, n_classes=10):
        super().__init__()
        self.img_proj = nn.Linear(img_dim, hidden_dim)
        self.txt_proj = nn.Linear(txt_dim, hidden_dim)
        
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=hidden_dim, nhead=4, dim_feedforward=512, batch_first=True
        )
        self.joint_encoder = nn.TransformerEncoder(encoder_layer, num_layers=2)
        self.classifier = nn.Linear(hidden_dim, n_classes)

    def forward(self, img_tokens, txt_tokens):
        # Project both to same dimension
        img = self.img_proj(img_tokens)    # [B, N_img, hidden]
        txt = self.txt_proj(txt_tokens)    # [B, N_txt, hidden]
        
        # EARLY FUSION: concatenate along sequence dimension
        combined = torch.cat([img, txt], dim=1)  # [B, N_img+N_txt, hidden]
        
        # Joint processing (all tokens attend to all tokens)
        output = self.joint_encoder(combined)
        
        # Pool and classify
        pooled = output.mean(dim=1)
        return self.classifier(pooled)


model = EarlyFusion()
img_tokens = torch.randn(2, 16, 128)   # 16 image patches
txt_tokens = torch.randn(2, 8, 128)    # 8 text tokens
out = model(img_tokens, txt_tokens)
print(f"Early Fusion output: {out.shape}")
count_parameters(model)

### Example 1: Early Fusion — Numerical Trace

Let's trace what happens when image patches and text tokens are concatenated and jointly processed:

**Setup:** 2 image patches + 3 text tokens, $d = 4$

```
Image patches:  [[0.5, -0.3, 0.8, 0.1],   ← patch_0 (contains cat)
                 [0.2,  0.7, -0.1, 0.4]]   ← patch_1 (contains sofa)

Text tokens:    [[0.9,  0.1, 0.3, -0.2],   ← "a"
                 [-0.1, 0.8, 0.6, 0.3],    ← "cat"
                 [0.4, -0.5, 0.2, 0.7]]    ← "sits"
```

After concatenation: **5 tokens × 4 dims** — the transformer sees ALL tokens at once. The attention matrix is $5 \times 5$, meaning "cat" (text) can directly attend to patch_0 (image with cat).

In [ ]:
# ============================================================
#  Example 1: Early Fusion — Step-by-Step Numerical Trace
# ============================================================

print("=" * 65)
print("  EARLY FUSION: Numerical Trace")
print("=" * 65)

# Image patches and text tokens
img_patches = torch.tensor([
    [0.5, -0.3, 0.8, 0.1],   # patch_0 (cat)
    [0.2,  0.7, -0.1, 0.4]   # patch_1 (sofa)
])
txt_tokens = torch.tensor([
    [0.9,  0.1, 0.3, -0.2],  # "a"
    [-0.1, 0.8, 0.6, 0.3],   # "cat"
    [0.4, -0.5, 0.2, 0.7]    # "sits"
])
all_names = ["img:cat", "img:sofa", "txt:a", "txt:cat", "txt:sits"]

# Concatenate (early fusion!)
combined = torch.cat([img_patches, txt_tokens], dim=0)
print(f"\nConcatenated sequence ({combined.shape[0]} tokens × {combined.shape[1]} dims):")
for i, name in enumerate(all_names):
    vals = ", ".join(f"{v:6.2f}" for v in combined[i])
    print(f"  {name:>10}: [{vals}]")

# Compute self-attention (simplified, single head)
d_k = combined.shape[1]
scores = combined @ combined.T  # raw dot products
scaled = scores / (d_k ** 0.5)
attn = torch.softmax(scaled, dim=-1)

print(f"\nSelf-attention matrix (who attends to whom):")
print(f"  {'':>10}", end="")
for name in all_names:
    print(f"  {name:>9}", end="")
print()
for i, name in enumerate(all_names):
    print(f"  {name:>10}", end="")
    for j in range(5):
        val = attn[i, j].item()
        marker = " ◀" if j == attn[i].argmax() else ""
        print(f"  {val:>7.3f}{marker:2}", end="")
    print()

print(f"\n💡 Key observations:")
for i, name in enumerate(all_names):
    top_idx = attn[i].argmax().item()
    print(f"  {name:>10} attends most to {all_names[top_idx]:<10} (weight={attn[i,top_idx]:.3f})")

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
im = ax.imshow(attn.numpy(), cmap='YlOrRd', vmin=0)
ax.set_xticks(range(5))
ax.set_yticks(range(5))
ax.set_xticklabels(all_names, rotation=45, ha='right', fontsize=9)
ax.set_yticklabels(all_names, fontsize=9)
for i in range(5):
    for j in range(5):
        color = 'white' if attn[i,j] > 0.3 else 'black'
        ax.text(j, i, f'{attn[i,j]:.2f}', ha='center', va='center', fontsize=9, color=color)
ax.set_title('Early Fusion: Full Attention Matrix\n(image + text tokens interact freely)', fontsize=12, fontweight='bold')
plt.colorbar(im, ax=ax)

# Show cross-modal vs intra-modal attention
ax = axes[1]
img_to_img = attn[:2, :2].mean().item()
img_to_txt = attn[:2, 2:].mean().item()
txt_to_img = attn[2:, :2].mean().item()
txt_to_txt = attn[2:, 2:].mean().item()

interactions = ['Img→Img', 'Img→Txt', 'Txt→Img', 'Txt→Txt']
values = [img_to_img, img_to_txt, txt_to_img, txt_to_txt]
colors = ['#E74C3C', '#9B59B6', '#9B59B6', '#3498DB']
ax.bar(interactions, values, color=colors, alpha=0.8, edgecolor='white', linewidth=2)
for i, (v, label) in enumerate(zip(values, interactions)):
    ax.text(i, v + 0.01, f'{v:.3f}', ha='center', fontsize=11, fontweight='bold')
ax.set_ylabel('Mean Attention Weight')
ax.set_title('Cross-Modal vs Intra-Modal Attention\n(purple = cross-modal interaction)', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.savefig('../assets/early_fusion_trace.png', dpi=150, bbox_inches='tight')
plt.show()

## 2. Late Fusion

**Idea:** Process each modality independently → combine only the final representations.  
**When:** Modalities are very different, or you want to reuse pretrained encoders.

### Late Fusion — Mathematical Forms

Each modality is encoded independently, then combined at the representation level:

| Operation | Formula | Parameters |
|-----------|---------|------------|
| **Concat** | $h = W [f_v; f_t] + b$ | $O((d_v + d_t) \cdot d_{\text{out}})$ |
| **Add** | $h = f_v + f_t$ | $O(1)$ — requires $d_v = d_t$ |
| **Multiply (Hadamard)** | $h = f_v \odot f_t$ | $O(1)$ — element-wise gating |

**Complexity advantage:** Late fusion costs $O(N^2 d + M^2 d)$ — each modality's encoder runs independently with its own self-attention. There is **no cross-modal attention** until the final combination step, making it cheap but limiting interaction depth.

In [ ]:
class LateFusion(nn.Module):
    """Process modalities separately, combine at the end."""
    def __init__(self, img_dim=128, txt_dim=128, hidden_dim=256, 
                 n_classes=10, fusion_type='concat'):
        super().__init__()
        self.fusion_type = fusion_type

        # Separate encoders (in practice these are pretrained)
        self.img_encoder = nn.Sequential(
            nn.Linear(img_dim, hidden_dim), nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim)
        )
        self.txt_encoder = nn.Sequential(
            nn.Linear(txt_dim, hidden_dim), nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim)
        )

        if fusion_type == 'concat':
            self.classifier = nn.Linear(hidden_dim * 2, n_classes)
        else:  # add or multiply
            self.classifier = nn.Linear(hidden_dim, n_classes)

    def forward(self, img_feat, txt_feat):
        img = self.img_encoder(img_feat)   # [B, hidden]
        txt = self.txt_encoder(txt_feat)   # [B, hidden]

        if self.fusion_type == 'concat':
            fused = torch.cat([img, txt], dim=-1)
        elif self.fusion_type == 'add':
            fused = img + txt
        elif self.fusion_type == 'multiply':
            fused = img * txt

        return self.classifier(fused)


# Compare all three late fusion types
img_feat = torch.randn(2, 128)
txt_feat = torch.randn(2, 128)

for ftype in ['concat', 'add', 'multiply']:
    model = LateFusion(fusion_type=ftype)
    out = model(img_feat, txt_feat)
    params = sum(p.numel() for p in model.parameters())
    print(f"Late Fusion ({ftype:8s}): output={out.shape}, params={params:,}")

## 3. Cross-Modal Attention Fusion (Most Powerful)

**Idea:** Let one modality attend to the other using cross-attention.  
This is what you already know! Q from one modality, K/V from another.

### Cross-Attention Formula

Cross-attention lets one modality **query** another. Text tokens ask: *"Which image patches are relevant to me?"*

$$\text{CrossAttn}(T, I) = \text{softmax}\left(\frac{(T W^Q)(I W^K)^\top}{\sqrt{d_k}}\right)(I W^V)$$

| Role | Source | Meaning |
|------|--------|---------|
| **Query (Q)** | Text tokens | "What am I looking for?" |
| **Key (K)** | Image patches | "What do I contain?" |
| **Value (V)** | Image patches | "What information do I provide?" |

Each text token learns **which image patches are relevant** — the word "cat" should attend strongly to patches containing the cat. This is the core mechanism in **LLaVA**, **Flamingo**, and **BLIP-2**, where a frozen vision encoder's patch tokens are cross-attended by an LLM's text tokens.

In [ ]:
class CrossModalFusion(nn.Module):
    """Cross-attention between image and text representations."""
    def __init__(self, dim=128, n_heads=4, n_layers=2, n_classes=10):
        super().__init__()
        
        # Text attends to image (text queries, image keys/values)
        self.cross_attn_layers = nn.ModuleList([
            nn.MultiheadAttention(dim, n_heads, batch_first=True)
            for _ in range(n_layers)
        ])
        self.norms = nn.ModuleList([
            nn.LayerNorm(dim) for _ in range(n_layers)
        ])
        self.ffns = nn.ModuleList([
            nn.Sequential(nn.Linear(dim, dim*4), nn.GELU(), nn.Linear(dim*4, dim))
            for _ in range(n_layers)
        ])
        self.ffn_norms = nn.ModuleList([
            nn.LayerNorm(dim) for _ in range(n_layers)
        ])
        
        self.classifier = nn.Linear(dim, n_classes)
        self._attn_weights = []

    def forward(self, txt_tokens, img_tokens, return_attention=False):
        self._attn_weights = []
        x = txt_tokens
        
        for cross_attn, norm, ffn, ffn_norm in zip(
            self.cross_attn_layers, self.norms, self.ffns, self.ffn_norms
        ):
            # Cross-attention: text (Q) attends to image (K, V)
            attn_out, attn_w = cross_attn(x, img_tokens, img_tokens)
            self._attn_weights.append(attn_w.detach())
            x = norm(x + attn_out)
            x = ffn_norm(x + ffn(x))
        
        pooled = x.mean(dim=1)
        logits = self.classifier(pooled)
        
        if return_attention:
            return logits, self._attn_weights
        return logits


model = CrossModalFusion(dim=128, n_heads=4, n_layers=2)
txt_tokens = torch.randn(1, 8, 128)    # 8 text tokens
img_tokens = torch.randn(1, 16, 128)   # 16 image patches

logits, attn_weights = model(txt_tokens, img_tokens, return_attention=True)
print(f"Output: {logits.shape}")
print(f"Attention weights: {len(attn_weights)} layers, each {attn_weights[0].shape}")

count_parameters(model)

In [ ]:
# Visualize cross-attention weights
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Cross-Attention: Text → Image', fontsize=16, fontweight='bold')

txt_labels = ['[CLS]', 'a', 'cat', 'on', 'a', 'sofa', '.', '[SEP]']
img_labels = [f'patch_{i}' for i in range(16)]

for i, (ax, weights) in enumerate(zip(axes, attn_weights)):
    w = weights[0].numpy()  # [txt_len, img_len]
    im = ax.imshow(w, cmap='YlOrRd', aspect='auto')
    ax.set_xticks(range(16))
    ax.set_xticklabels(img_labels, rotation=45, fontsize=8)
    ax.set_yticks(range(8))
    ax.set_yticklabels(txt_labels)
    ax.set_title(f'Layer {i+1} Cross-Attention', fontsize=12)
    ax.set_xlabel('Image Patches (Keys)')
    ax.set_ylabel('Text Tokens (Queries)')
    plt.colorbar(im, ax=ax, shrink=0.8)

plt.tight_layout()
plt.savefig('../assets/cross_attention_weights.png', dpi=150, bbox_inches='tight')
plt.show()
print("Each row shows which image patches a text token pays attention to.")
print("After training, 'cat' should attend to the patches containing the cat!")

### Example 2: Cross-Attention — Complete Numerical Walkthrough

Cross-attention is the most important fusion mechanism. Let's trace it step by step:

**Setup:** Text token "cat" queries image patches to find the relevant visual region.

| | dim_0 | dim_1 | dim_2 | dim_3 |
|---|------|-------|-------|-------|
| **Q** (from "cat") | 0.8 | -0.3 | 0.5 | 0.1 |
| **K** (from patch_cat) | 0.7 | -0.2 | 0.6 | 0.0 |
| **K** (from patch_sky) | -0.1 | 0.5 | -0.3 | 0.8 |
| **K** (from patch_sofa) | 0.3 | 0.1 | 0.2 | 0.4 |

The word "cat" should attend strongly to the patch containing the cat image.

In [ ]:
# ============================================================
#  Example 2: Cross-Attention — Numerical Walkthrough
# ============================================================

print("=" * 65)
print("  CROSS-ATTENTION: Text → Image Numerical Trace")
print("=" * 65)

# Text query: "cat" wants to find relevant image patches
Q_cat = torch.tensor([[0.8, -0.3, 0.5, 0.1]])  # query from text "cat"

# Image keys: 4 patches from the image
K_patches = torch.tensor([
    [0.7, -0.2, 0.6, 0.0],    # patch with cat
    [-0.1, 0.5, -0.3, 0.8],   # patch with sky
    [0.3, 0.1, 0.2, 0.4],     # patch with sofa
    [-0.4, 0.3, -0.1, 0.6]    # patch with floor
])

V_patches = torch.tensor([
    [1.0, 0.0, 0.0, 0.0],    # cat features
    [0.0, 1.0, 0.0, 0.0],    # sky features
    [0.0, 0.0, 1.0, 0.0],    # sofa features
    [0.0, 0.0, 0.0, 1.0]     # floor features
])

patch_names = ["cat-patch", "sky-patch", "sofa-patch", "floor-patch"]
d_k = 4

# Step 1: Q @ K^T
scores = Q_cat @ K_patches.T  # [1, 4]
print(f"\nStep 1: Raw scores (Q_cat @ K_patches^T):")
for i, name in enumerate(patch_names):
    # Show dot product breakdown
    terms = [f"({Q_cat[0,d]:.1f})({K_patches[i,d]:.1f})" for d in range(d_k)]
    total = scores[0, i].item()
    print(f"  Q·K[{name:>11}] = {' + '.join(terms)} = {total:.3f}")

# Step 2: Scale
scaled = scores / (d_k ** 0.5)
print(f"\nStep 2: Scaled scores (÷ √{d_k} = ÷ {d_k**0.5:.2f}):")
for i, name in enumerate(patch_names):
    print(f"  {name:>11}: {scores[0,i]:.3f} → {scaled[0,i]:.3f}")

# Step 3: Softmax
attn = torch.softmax(scaled, dim=-1)
print(f"\nStep 3: Attention weights (softmax):")
for i, name in enumerate(patch_names):
    bar = "█" * int(attn[0,i].item() * 40)
    print(f"  {name:>11}: {attn[0,i]:.4f}  {bar}")

# Step 4: Weighted sum of values
output = attn @ V_patches
print(f"\nStep 4: Output = weighted sum of Value vectors:")
print(f"  = {attn[0,0]:.3f}×V[cat] + {attn[0,1]:.3f}×V[sky] + {attn[0,2]:.3f}×V[sofa] + {attn[0,3]:.3f}×V[floor]")
print(f"  = [{output[0,0]:.3f}, {output[0,1]:.3f}, {output[0,2]:.3f}, {output[0,3]:.3f}]")
print(f"\n  ✅ Output is dominated by cat features ({attn[0,0]:.1%} cat)")
print(f"     → 'cat' token successfully found the cat patch!")

# Visualize
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Attention weights
ax = axes[0]
colors = ['#E74C3C', '#3498DB', '#2ECC71', '#F39C12']
bars = ax.bar(patch_names, attn[0].numpy(), color=colors, alpha=0.8)
ax.set_ylabel('Attention Weight')
ax.set_title('Cross-Attention: "cat" → patches\n(correctly attends to cat-patch)', fontsize=12, fontweight='bold')
for bar, val in zip(bars, attn[0].numpy()):
    ax.text(bar.get_x() + bar.get_width()/2, val + 0.02, f'{val:.3f}', 
            ha='center', fontsize=11, fontweight='bold')

# Output vector composition
ax = axes[1]
components = ['cat-feat', 'sky-feat', 'sofa-feat', 'floor-feat']
ax.bar(components, output[0].numpy(), color=colors, alpha=0.8)
ax.set_ylabel('Value')
ax.set_title('Output Vector Composition\n(weighted mix of visual features)', fontsize=12, fontweight='bold')

# Full cross-attention with multiple text queries
ax = axes[2]
Q_full = torch.tensor([
    [0.8, -0.3, 0.5, 0.1],   # "cat"
    [-0.1, 0.5, -0.3, 0.8],  # "sky"  
    [0.3, 0.1, 0.2, 0.4]     # "sofa"
])
txt_names = ['"cat"', '"sky"', '"sofa"']
full_scores = Q_full @ K_patches.T / (d_k ** 0.5)
full_attn = torch.softmax(full_scores, dim=-1)
im = ax.imshow(full_attn.numpy(), cmap='YlOrRd', vmin=0, vmax=1)
ax.set_xticks(range(4))
ax.set_yticks(range(3))
ax.set_xticklabels(patch_names, rotation=30, ha='right', fontsize=9)
ax.set_yticklabels(txt_names, fontsize=10)
for i in range(3):
    for j in range(4):
        color = 'white' if full_attn[i,j] > 0.3 else 'black'
        ax.text(j, i, f'{full_attn[i,j]:.2f}', ha='center', va='center', fontsize=10, color=color)
ax.set_title('Full Cross-Attention Matrix\n(each word finds its patch)', fontsize=12, fontweight='bold')
plt.colorbar(im, ax=ax)

plt.tight_layout()
plt.savefig('../assets/cross_attention_trace.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Gated Fusion (Adaptive Weighting)

**Idea:** Learn to weight how much each modality contributes.

### Gated Fusion — The Math

The gate learns an **adaptive weight** for each sample based on both modalities:

$$g = \sigma(W_g [f_v; f_t] + b_g)$$

where $\sigma$ is the sigmoid function ($g \in [0, 1]$), and the fused representation is:

$$h = g \odot f_v + (1 - g) \odot f_t$$

**Why this matters:**
- When the image is **noisy or missing**, the gate can shift toward text ($g \to 0$)
- When text is **ambiguous**, the gate can rely on vision ($g \to 1$)
- Unlike fixed-weight late fusion (add/multiply), gating is **input-dependent** — each sample gets its own modality balance

This is especially useful in real-world settings where one modality may be corrupted (blurry image, OCR errors in text).

In [ ]:
class GatedFusion(nn.Module):
    """Learned gating to control modality contribution."""
    def __init__(self, dim=128, n_classes=10):
        super().__init__()
        self.gate = nn.Sequential(
            nn.Linear(dim * 2, dim),
            nn.ReLU(),
            nn.Linear(dim, 1),
            nn.Sigmoid()  # gate value between 0 and 1
        )
        self.classifier = nn.Linear(dim, n_classes)

    def forward(self, img_feat, txt_feat):
        combined = torch.cat([img_feat, txt_feat], dim=-1)
        gate_value = self.gate(combined)  # [B, 1] between 0 and 1
        
        # Weighted combination
        fused = gate_value * img_feat + (1 - gate_value) * txt_feat
        return self.classifier(fused), gate_value


model = GatedFusion()
img_feat = torch.randn(8, 128)
txt_feat = torch.randn(8, 128)

logits, gates = model(img_feat, txt_feat)

# Visualize gate values
fig, ax = plt.subplots(figsize=(8, 4))
gate_vals = gates.detach().numpy().flatten()
bars = ax.bar(range(len(gate_vals)), gate_vals, color='#9B59B6', alpha=0.7)
ax.axhline(y=0.5, color='red', linestyle='--', label='Equal weighting')
ax.set_xlabel('Sample')
ax.set_ylabel('Gate Value')
ax.set_title('Gated Fusion: Image Contribution Weight\n(>0.5 = more image, <0.5 = more text)', 
             fontsize=12, fontweight='bold')
ax.legend()
ax.set_ylim(0, 1)
plt.tight_layout()
plt.show()

### Bilinear Fusion

Beyond additive and multiplicative combinations, **bilinear fusion** captures *multiplicative interactions* between modalities:

$$h = f_v^\top W f_t$$

where $W \in \mathbb{R}^{d_v \times d_t \times d_{\text{out}}}$ is a 3D weight tensor. This lets the model learn that certain visual features *combined with* certain text features produce a strong signal — e.g., "red" + red-patch activation.

**The problem:** Full bilinear fusion requires $d_v \times d_t \times d_{\text{out}}$ parameters. For $d_v = d_t = d_{\text{out}} = 768$, that's **450M parameters** just for the fusion layer!

**Low-rank approximation (MLB — Multimodal Low-rank Bilinear):**

$$h = (U^\top f_v) \odot (V^\top f_t)$$

where $U, V \in \mathbb{R}^{d \times k}$ with rank $k \ll d$. Parameter count drops to **$2 \times d \times k$** (e.g., $k=256$ → ~393K params). This is used in VQA models like **MCB** and **MUTAN** to capture rich cross-modal interactions without the full tensor cost.

In [ ]:
# ============================================================
#  Example 3: Bilinear Fusion — Full vs Low-Rank Implementation
# ============================================================

print("=" * 65)
print("  BILINEAR FUSION: Full Rank vs MLB (Low-Rank)")
print("=" * 65)

class FullBilinearFusion(nn.Module):
    """Full bilinear: h = v^T W t (very expensive!)"""
    def __init__(self, d_v=128, d_t=128, d_out=64):
        super().__init__()
        self.W = nn.Parameter(torch.randn(d_v, d_t, d_out) * 0.01)
    
    def forward(self, v, t):
        # v: [B, d_v], t: [B, d_t]
        # Einstein summation: batch, d_v, d_t, d_out
        return torch.einsum('bi,ijk,bj->bk', v, self.W, t)

class MLBFusion(nn.Module):
    """Multimodal Low-rank Bilinear: h = (U^T v) ⊙ (V^T t)"""
    def __init__(self, d_v=128, d_t=128, d_out=64, rank=32):
        super().__init__()
        self.U = nn.Linear(d_v, rank)
        self.V = nn.Linear(d_t, rank)
        self.out = nn.Linear(rank, d_out)
    
    def forward(self, v, t):
        return self.out(torch.relu(self.U(v)) * torch.relu(self.V(t)))

# Compare parameter counts
d_v, d_t, d_out = 128, 128, 64

full = FullBilinearFusion(d_v, d_t, d_out)
mlb_32 = MLBFusion(d_v, d_t, d_out, rank=32)
mlb_8 = MLBFusion(d_v, d_t, d_out, rank=8)

params_full = sum(p.numel() for p in full.parameters())
params_mlb32 = sum(p.numel() for p in mlb_32.parameters())
params_mlb8 = sum(p.numel() for p in mlb_8.parameters())

print(f"\n{'Method':<25} {'Parameters':>12} {'Compression':>12}")
print("─" * 50)
print(f"{'Full Bilinear':<25} {params_full:>12,} {'1.0x':>12}")
print(f"{'MLB (rank=32)':<25} {params_mlb32:>12,} {f'{params_full/params_mlb32:.1f}x':>12}")
print(f"{'MLB (rank=8)':<25} {params_mlb8:>12,} {f'{params_full/params_mlb8:.1f}x':>12}")

# Test both
v = torch.randn(4, d_v)
t = torch.randn(4, d_t)

out_full = full(v, t)
out_mlb = mlb_32(v, t)

print(f"\nOutput shapes: Full={out_full.shape}, MLB={out_mlb.shape}")
print(f"Output norms: Full={out_full.norm():.3f}, MLB={out_mlb.norm():.3f}")

# Scaling to realistic dimensions
print(f"\n📊 At ViT-Base scale (d=768):")
for d in [768]:
    full_params = d * d * d  # Full bilinear tensor
    for rank in [8, 32, 64, 128, 256]:
        mlb_params = 2 * d * rank + rank * d  # U, V, out
        print(f"  Full: {full_params/1e6:.0f}M params | MLB(rank={rank:>3}): {mlb_params/1e6:.1f}M params | Compression: {full_params/mlb_params:.0f}x")

# Visualize comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Parameter comparison
ax = axes[0]
methods = ['Full\nBilinear', 'MLB\n(rank=32)', 'MLB\n(rank=8)', 'Late\n(concat)', 'Gated']
params_list = [params_full, params_mlb32, params_mlb8, 
               sum(p.numel() for p in LateFusion().parameters()),
               sum(p.numel() for p in GatedFusion().parameters())]
colors = ['#E74C3C', '#3498DB', '#2ECC71', '#F39C12', '#9B59B6']
ax.bar(methods, [p/1000 for p in params_list], color=colors, alpha=0.8)
ax.set_ylabel('Parameters (K)')
ax.set_title('Fusion Method Parameter Comparison\n(smaller = more efficient)', fontsize=12, fontweight='bold')
for i, p in enumerate(params_list):
    ax.text(i, p/1000 + 10, f'{p:,}', ha='center', fontsize=9)

# Interaction expressiveness
ax = axes[1]
expressiveness = [10, 8, 6, 3, 5]  # relative scores
ax.bar(methods, expressiveness, color=colors, alpha=0.8)
ax.set_ylabel('Interaction Expressiveness (relative)')
ax.set_title('Expressiveness vs Efficiency\n(bilinear captures multiplicative interactions)', fontsize=12, fontweight='bold')
ax.set_ylim(0, 12)

plt.tight_layout()
plt.savefig('../assets/bilinear_fusion_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## Fusion Strategy Comparison

| Strategy | Pros | Cons | Computational Cost | Best For |
|----------|------|------|--------------------|----------|
| **Early** | Maximum interaction | Expensive, needs same structure | $O((N+M)^2 d)$ — quadratic in joint sequence | Similar modalities |
| **Late** | Simple, reuse encoders | Limited interaction | $O(N^2 d + M^2 d)$ — independent encoders | Quick baseline |
| **Cross-Attn** | Rich interaction, interpretable | More params | $O(N \cdot M \cdot d)$ per layer | Most tasks (SOTA) |
| **Gated** | Adaptive, lightweight | Simple weighting | $O(d)$ — negligible overhead | When modalities vary in quality |
| **Bilinear** | Captures multiplicative interactions | Full rank is parameter-heavy | $O(d_v d_t k)$ with low-rank (MLB) | VQA, fine-grained matching |

> **Rule of thumb:** Early fusion scales poorly as sequence length grows ($N+M$ can exceed 500+ tokens in LLaVA-style models). Cross-attention gives the best interaction-to-cost ratio for vision-language tasks.

---
**Next:** Module 02 - Vision-Language Models (CLIP from scratch)

### Example 4: Real-World — Choosing the Right Fusion Strategy

Different applications need different fusion strategies. Here's a decision guide with real examples:

| Application | Recommended Fusion | Why |
|------------|-------------------|-----|
| **Visual QA** | Cross-Attention | Questions selectively attend to relevant image regions |
| **Image-Text Retrieval** | Late Fusion (CLIP) | Need separate embeddings for fast indexing |
| **Video Captioning** | Early Fusion | Dense temporal-visual interaction needed |
| **Medical Diagnosis** | Gated Fusion | Image quality varies; gate downweights bad scans |
| **Visual Grounding** | Cross-Attention + Bilinear | Fine-grained patch-word matching |

In [ ]:
# ============================================================
#  Example 4: Side-by-Side Fusion Strategy Comparison
# ============================================================

print("=" * 65)
print("  FUSION STRATEGY COMPARISON — Training on Same Data")
print("=" * 65)

# Create a simple classification task with both image and text features
torch.manual_seed(42)

N_SAMPLES = 500
N_CLASSES = 5
D_FEAT = 64

# Generate features where BOTH modalities carry signal
X_img = torch.randn(N_SAMPLES, D_FEAT)
X_txt = torch.randn(N_SAMPLES, D_FEAT)
labels = torch.randint(0, N_CLASSES, (N_SAMPLES,))

# Inject class signal into both modalities
for i in range(N_SAMPLES):
    c = labels[i].item()
    X_img[i, c*10:(c+1)*10] += 2.0  # image signal in different region per class
    X_txt[i, c*10:(c+1)*10] += 1.5  # text signal (weaker)

# Define simple fusion models
class SimpleFusion(nn.Module):
    def __init__(self, fusion_type='concat', d=64, n_classes=5):
        super().__init__()
        self.fusion_type = fusion_type
        
        if fusion_type == 'concat':
            self.head = nn.Sequential(nn.Linear(d*2, 128), nn.ReLU(), nn.Linear(128, n_classes))
        elif fusion_type == 'add':
            self.head = nn.Sequential(nn.Linear(d, 128), nn.ReLU(), nn.Linear(128, n_classes))
        elif fusion_type == 'gated':
            self.gate = nn.Sequential(nn.Linear(d*2, d), nn.Sigmoid())
            self.head = nn.Sequential(nn.Linear(d, 128), nn.ReLU(), nn.Linear(128, n_classes))
        elif fusion_type == 'bilinear':
            self.U = nn.Linear(d, 32)
            self.V = nn.Linear(d, 32)
            self.head = nn.Sequential(nn.Linear(32, 128), nn.ReLU(), nn.Linear(128, n_classes))
    
    def forward(self, img, txt):
        if self.fusion_type == 'concat':
            return self.head(torch.cat([img, txt], dim=-1))
        elif self.fusion_type == 'add':
            return self.head(img + txt)
        elif self.fusion_type == 'gated':
            g = self.gate(torch.cat([img, txt], dim=-1))
            return self.head(g * img + (1-g) * txt)
        elif self.fusion_type == 'bilinear':
            return self.head(torch.relu(self.U(img)) * torch.relu(self.V(txt)))

# Train each fusion type
results = {}
for ftype in ['concat', 'add', 'gated', 'bilinear']:
    model = SimpleFusion(ftype, d=D_FEAT, n_classes=N_CLASSES)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    
    losses = []
    accs = []
    for epoch in range(100):
        # Mini-batch training
        perm = torch.randperm(N_SAMPLES)
        epoch_loss = 0
        correct = 0
        for start in range(0, N_SAMPLES, 64):
            idx = perm[start:start+64]
            logits = model(X_img[idx], X_txt[idx])
            loss = F.cross_entropy(logits, labels[idx])
            
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            
            epoch_loss += loss.item()
            correct += (logits.argmax(-1) == labels[idx]).sum().item()
        
        losses.append(epoch_loss / (N_SAMPLES // 64))
        accs.append(correct / N_SAMPLES * 100)
    
    n_params = sum(p.numel() for p in model.parameters())
    results[ftype] = {'losses': losses, 'accs': accs, 'params': n_params}
    print(f"  {ftype:>10}: Final acc = {accs[-1]:.1f}%  |  params = {n_params:,}")

# Visualize
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
colors_map = {'concat': '#E74C3C', 'add': '#3498DB', 'gated': '#9B59B6', 'bilinear': '#2ECC71'}

# Loss curves
ax = axes[0]
for ftype, data in results.items():
    ax.plot(data['losses'], label=ftype, color=colors_map[ftype], linewidth=2)
ax.set_xlabel('Epoch')
ax.set_ylabel('Loss')
ax.set_title('Training Loss by Fusion Type', fontsize=13, fontweight='bold')
ax.legend()

# Accuracy curves
ax = axes[1]
for ftype, data in results.items():
    ax.plot(data['accs'], label=ftype, color=colors_map[ftype], linewidth=2)
ax.set_xlabel('Epoch')
ax.set_ylabel('Accuracy (%)')
ax.set_title('Training Accuracy by Fusion Type', fontsize=13, fontweight='bold')
ax.legend()

# Final comparison
ax = axes[2]
ftypes = list(results.keys())
final_accs = [results[f]['accs'][-1] for f in ftypes]
params = [results[f]['params'] for f in ftypes]
colors_list = [colors_map[f] for f in ftypes]

bars = ax.bar(ftypes, final_accs, color=colors_list, alpha=0.8)
for bar, acc, p in zip(bars, final_accs, params):
    ax.text(bar.get_x() + bar.get_width()/2, acc + 0.5, 
            f'{acc:.1f}%\n({p:,} params)', ha='center', fontsize=9)
ax.set_ylabel('Final Accuracy (%)')
ax.set_title('Final Accuracy Comparison\n(with parameter count)', fontsize=13, fontweight='bold')
ax.set_ylim(0, 105)

plt.tight_layout()
plt.savefig('../assets/fusion_comparison_training.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\n🏆 Results Summary:")
best = max(results.items(), key=lambda x: x[1]['accs'][-1])
most_efficient = min(results.items(), key=lambda x: x[1]['params'])
print(f"  Best accuracy: {best[0]} ({best[1]['accs'][-1]:.1f}%)")
print(f"  Most efficient: {most_efficient[0]} ({most_efficient[1]['params']:,} params)")

---

## 📚 References & Further Reading

### Papers
- **Multimodal Learning with Transformers: A Survey** — Xu et al., 2023 — [arXiv:2206.06488](https://arxiv.org/abs/2206.06488) — Comprehensive fusion strategies survey
- **ViLBERT: Pretraining Task-Agnostic Visiolinguistic Representations** — Lu et al., 2019 — [arXiv:1908.02265](https://arxiv.org/abs/1908.02265) — Co-attention fusion
- **LXMERT: Learning Cross-Modality Encoder Representations** — Tan & Bansal, 2019 — [arXiv:1908.07490](https://arxiv.org/abs/1908.07490) — Cross-modal Transformer
- **Multimodal Compact Bilinear Pooling (MCB)** — Fukui et al., 2016 — [arXiv:1606.01847](https://arxiv.org/abs/1606.01847) — Bilinear fusion for VQA
- **MUTAN: Multimodal Tucker Fusion** — Ben-Younes et al., 2017 — [arXiv:1705.06676](https://arxiv.org/abs/1705.06676) — Low-rank bilinear fusion
- **Perceiver: General Perception with Iterative Attention** — Jaegle et al., 2021 — [arXiv:2103.03206](https://arxiv.org/abs/2103.03206) — Cross-attention for arbitrary modalities

### Blog Posts & Cheat Sheets
- 🔗 [Multimodal Fusion Strategies Explained](https://neptune.ai/blog/multimodal-learning) — Neptune.ai — Visual comparison of fusion types
- 🔗 [Cross-Attention Explained](https://vaclavkosar.com/ml/cross-attention-in-transformer-architecture) — Detailed cross-attention walkthrough
- 🔗 [Lilian Weng: The Transformer Family v2](https://lilianweng.github.io/posts/2023-01-27-the-transformer-family-v2/) — Comprehensive attention mechanism catalog
- 🔗 [Papers With Code: Multimodal Fusion](https://paperswithcode.com/task/multimodal-fusion) — Benchmark and leaderboard